# 📦 15 — LCC FASD 다운로드 & 구글 드라이브 배치
> **목적:** Kaggle의 LCC FASD 데이터셋에서 Real(Live) 이미지만 추출 →  
> 웹캠 도메인 갭 해결을 위한 Fine-tuning 데이터로 활용  
> **출력:** `data/webcam_live/` — 얼굴 크롭 완료된 Live 이미지

---
## ✅ 실행 전 체크리스트
- [ ] Cell 1: 구글 드라이브 마운트
- [ ] Cell 2: Kaggle API 키 업로드
- [ ] Cell 3: LCC FASD 다운로드
- [ ] Cell 4: 데이터셋 구조 확인
- [ ] Cell 5: Real 이미지만 추출 + 얼굴 크롭
- [ ] Cell 6: 최종 확인 & 수치 분포 점검

## Cell 1 — 구글 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE       = '/content/drive/MyDrive/face-anti-spoofing'
DATA_DIR   = f'{BASE}/data'
WEBCAM_DIR = f'{DATA_DIR}/webcam_live'   # ← 최종 출력 폴더
WORK_DIR   = '/content/lcc_fasd'         # Colab 임시 작업 폴더

os.makedirs(WEBCAM_DIR, exist_ok=True)
os.makedirs(WORK_DIR,   exist_ok=True)

# 기존 cropped 폴더 구조 확인
print('=== 기존 data/ 폴더 구조 ===')
for cat in ['live', 'print', 'replay', 'mask', 'webcam_live']:
    d = f'{DATA_DIR}/cropped/{cat}'
    n = len(os.listdir(d)) if os.path.exists(d) else 0
    print(f'  {cat:<15}: {n}장')

print(f'\n✅ 작업 경로')
print(f'  Colab 임시: {WORK_DIR}')
print(f'  Drive 출력: {WEBCAM_DIR}')

## Cell 2 — Kaggle API 키 설정

> **사전 준비:**  
> 1. https://www.kaggle.com → 우측 상단 프로필 → **Settings**  
> 2. **API** 섹션 → **Create New Token** → `kaggle.json` 다운로드  
> 3. 아래 셀 실행 후 뜨는 파일 업로드 버튼으로 `kaggle.json` 업로드

In [ ]:
from google.colab import files
import os, shutil

print('📂 kaggle.json 파일을 업로드해주세요...')
uploaded = files.upload()   # kaggle.json 선택

# Kaggle 인증 디렉토리에 배치
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Kaggle CLI 설치
!pip install -q kaggle

# 인증 확인
!kaggle datasets list --search 'lcc-fasd' 2>&1 | head -5
print('\n✅ Kaggle API 설정 완료')

## Cell 3 — LCC FASD 다운로드

> 전체 크기 약 **2~3GB** (Real + Fake 합산)  
> Colab 임시 디렉토리에 받은 뒤, Real만 Drive로 이동 (Drive 용량 절약)

In [ ]:
import os

os.chdir(WORK_DIR)

print('📥 LCC FASD 다운로드 중... (2~3GB, 수 분 소요)')
!kaggle datasets download -d faber24/lcc-fasd --unzip -p {WORK_DIR}

print('\n=== 다운로드된 파일/폴더 구조 ===')
for item in sorted(os.listdir(WORK_DIR)):
    full = os.path.join(WORK_DIR, item)
    if os.path.isdir(full):
        n = sum(len(files) for _, _, files in os.walk(full))
        print(f'  📁 {item}/  ({n}개 파일)')
    else:
        size_mb = os.path.getsize(full) / 1024 / 1024
        print(f'  📄 {item}  ({size_mb:.1f} MB)')

## Cell 4 — 데이터셋 구조 상세 확인

> LCC FASD 폴더 구조:  
> ```
> LCC_FASD/
> ├── LCC_FASD_development/
> │   ├── real/       ← 웹캠/모바일 촬영 Live 얼굴
> │   └── spoof/      ← Print/Replay 공격
> └── LCC_FASD_evaluation/
>     ├── real/
>     └── spoof/
> ```

In [ ]:
import os
from pathlib import Path

# 실제 구조 탐색 (다운로드된 폴더명이 다를 수 있음)
print('=== 전체 폴더 트리 (depth 3) ===')
for root, dirs, files in os.walk(WORK_DIR):
    depth = root.replace(WORK_DIR, '').count(os.sep)
    if depth > 3:
        continue
    indent = '  ' * depth
    folder_name = os.path.basename(root)
    n_files = len(files)
    print(f'{indent}📁 {folder_name}/  [{n_files}파일]')

# real 폴더 자동 탐색
print('\n=== real/ 폴더 위치 자동 탐색 ===')
real_dirs = []
for root, dirs, files in os.walk(WORK_DIR):
    if os.path.basename(root).lower() in ['real', 'live', 'genuine']:
        n = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        real_dirs.append((root, n))
        print(f'  ✅ {root}  ({n}장)')

print(f'\n총 real 폴더: {len(real_dirs)}개')
print(f'총 real 이미지: {sum(n for _, n in real_dirs)}장')

## Cell 5 — Real 이미지 얼굴 크롭 & Drive 저장

> **전략:**
> - Real 이미지에서 Haar Cascade로 얼굴 영역 크롭 (기존 파이프라인과 동일)
> - 224×224로 리사이즈
> - 최대 **200장만** 추출 (기존 CelebA Live 1500장과 균형 고려)
> - 실패한 이미지(얼굴 미탐지)는 건너뜀

In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path
import random

# ── Haar Cascade 로드 (기존 파이프라인과 동일) ──────────────────
CASCADE_PATH = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(CASCADE_PATH)
print(f'Haar Cascade 로드: {CASCADE_PATH}')

# ── 설정 ────────────────────────────────────────────────────────
TARGET_SIZE  = (224, 224)
MAX_IMAGES   = 200          # 최대 추출 장수
MARGIN_RATIO = 0.2          # 얼굴 박스 여유 비율
OUTPUT_DIR   = WEBCAM_DIR   # Drive의 webcam_live/


def crop_face(img_bgr, margin=MARGIN_RATIO):
    """Haar Cascade로 얼굴 크롭 → 224×224. 실패 시 None 반환."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
    )
    if len(faces) == 0:
        return None
    # 가장 큰 얼굴 선택
    x, y, w, h = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)[0]
    H, W = img_bgr.shape[:2]
    mx, my = int(w * margin), int(h * margin)
    x1 = max(0, x - mx)
    y1 = max(0, y - my)
    x2 = min(W, x + w + mx)
    y2 = min(H, y + h + my)
    face = img_bgr[y1:y2, x1:x2]
    return cv2.resize(face, TARGET_SIZE)


# ── real 이미지 목록 수집 ────────────────────────────────────────
all_real_imgs = []
for root, dirs, files in os.walk(WORK_DIR):
    if os.path.basename(root).lower() in ['real', 'live', 'genuine']:
        for f in files:
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_real_imgs.append(os.path.join(root, f))

random.seed(42)
random.shuffle(all_real_imgs)
print(f'총 real 이미지: {len(all_real_imgs)}장')
print(f'목표 추출: 최대 {MAX_IMAGES}장')

# ── 얼굴 크롭 & 저장 ─────────────────────────────────────────────
saved = 0
skipped = 0

for i, img_path in enumerate(all_real_imgs):
    if saved >= MAX_IMAGES:
        break

    raw  = np.fromfile(img_path, dtype=np.uint8)
    img  = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    if img is None:
        skipped += 1
        continue

    face = crop_face(img)
    if face is None:
        skipped += 1
        continue

    out_path = os.path.join(OUTPUT_DIR, f'lcc_real_{saved:04d}.jpg')
    cv2.imencode('.jpg', face)[1].tofile(out_path)
    saved += 1

    if saved % 20 == 0:
        print(f'  진행: {saved}/{MAX_IMAGES}장 저장, {skipped}장 스킵 (총 시도 {i+1}장)')

print(f'\n✅ 완료: {saved}장 저장, {skipped}장 스킵')
print(f'저장 위치: {OUTPUT_DIR}')

## Cell 6 — 최종 확인 & Laplacian/FFT 수치 분포 점검

> **핵심 체크:** LCC FASD Real 이미지의 Laplacian/FFT 수치가  
> 우리 웹캠 수치(Laplacian≈601, FFT≈1237)와 유사한지 확인

| 카테고리 | Laplacian | FFT High | 비고 |
|---------|-----------|----------|------|
| CelebA Live | 383 | 1134 | 학습 데이터 |
| 우리 웹캠 Live | 601 | 1237 | 오탐 원인 |
| **LCC FASD Real** | ??? | ??? | ← 지금 측정 |

In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt

# ── 수치 측정 ────────────────────────────────────────────────────
def measure_stats(img_bgr):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lap  = cv2.Laplacian(gray, cv2.CV_64F).var()

    gray_f = gray.astype(np.float32)
    f = np.fft.fftshift(np.fft.fft2(gray_f))
    mag = np.abs(f)
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    r = min(h, w) // 6
    Y, X = np.ogrid[:h, :w]
    mask = (Y - cy)**2 + (X - cx)**2 > r**2
    fft_high = float(mag[mask].mean())
    return float(lap), fft_high

laps, ffts = [], []
imgs = sorted(Path(WEBCAM_DIR).glob('*.jpg'))[:100]  # 최대 100장 측정

for p in imgs:
    raw = np.fromfile(str(p), dtype=np.uint8)
    img = cv2.imdecode(raw, cv2.IMREAD_COLOR)
    if img is not None:
        l, f = measure_stats(img)
        laps.append(l)
        ffts.append(f)

print('=== LCC FASD Real 이미지 수치 분포 ===')
print(f'  측정 장수:      {len(laps)}장')
print(f'  Laplacian 평균: {np.mean(laps):.1f}  (우리 웹캠: 601 | CelebA: 383)')
print(f'  Laplacian 중앙: {np.median(laps):.1f}')
print(f'  FFT High 평균:  {np.mean(ffts):.1f}  (우리 웹캠: 1237 | CelebA: 1134)')
print(f'  FFT High 중앙:  {np.median(ffts):.1f}')

# ── 기준값 비교 시각화 ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('LCC FASD Real — 수치 분포 vs 기준값', fontsize=13)

CELEBA_LAP,  WEBCAM_LAP  = 383, 601
CELEBA_FFT,  WEBCAM_FFT  = 1134, 1237

axes[0].hist(laps, bins=30, color='steelblue', alpha=0.7, label='LCC FASD Real')
axes[0].axvline(CELEBA_LAP,  color='green',  ls='--', lw=2, label=f'CelebA Live ({CELEBA_LAP})')
axes[0].axvline(WEBCAM_LAP,  color='red',    ls='--', lw=2, label=f'우리 웹캠 ({WEBCAM_LAP})')
axes[0].axvline(np.mean(laps), color='blue', ls='-',  lw=2, label=f'LCC 평균 ({np.mean(laps):.0f})')
axes[0].set_title('Laplacian Variance 분포')
axes[0].set_xlabel('Laplacian Var')
axes[0].legend(fontsize=8)

axes[1].hist(ffts, bins=30, color='darkorange', alpha=0.7, label='LCC FASD Real')
axes[1].axvline(CELEBA_FFT,  color='green',  ls='--', lw=2, label=f'CelebA Live ({CELEBA_FFT})')
axes[1].axvline(WEBCAM_FFT,  color='red',    ls='--', lw=2, label=f'우리 웹캠 ({WEBCAM_FFT})')
axes[1].axvline(np.mean(ffts), color='blue', ls='-',  lw=2, label=f'LCC 평균 ({np.mean(ffts):.0f})')
axes[1].set_title('FFT High Frequency 분포')
axes[1].set_xlabel('FFT High Mean')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{BASE}/reports/lcc_fasd_stats.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 판정 ─────────────────────────────────────────────────────────
print('\n=== 도메인 적합성 판정 ===')
lap_mean = np.mean(laps)
if lap_mean > 450:
    print(f'  ✅ Laplacian {lap_mean:.0f} → 웹캠 도메인과 유사! Fine-tuning 적합')
elif lap_mean > 350:
    print(f'  ⚠️ Laplacian {lap_mean:.0f} → CelebA와 웹캠 중간. 사용 가능하나 효과 제한적')
else:
    print(f'  ❌ Laplacian {lap_mean:.0f} → CelebA와 유사. 도메인 갭 해결 어려움')

print(f'\n총 저장 이미지: {len(list(Path(WEBCAM_DIR).glob("*.jpg")))}장')
print('\n✅ 다음 단계: 16_webcam_v2_finetune.ipynb 에서 Fine-tuning 진행')

---
## ⏭️ 다음 단계 (16_webcam_v2_finetune.ipynb)

### Cell 6 결과에 따른 분기

| LCC Laplacian 평균 | 판정 | 다음 액션 |
|-------------------|------|----------|
| > 450 | ✅ 웹캠 도메인 유사 | 바로 Fine-tuning 진행 |
| 350~450 | ⚠️ 중간 | Fine-tuning + 본인 웹캠 51장 혼합 |
| < 350 | ❌ CelebA와 유사 | 본인 웹캠 51장만 사용하는 기존 방식으로 복귀 |

### Fine-tuning 데이터 구성 계획
```
webcam_live/     LCC FASD Real 200장  ← 이 노트북 출력
    +
기존 본인 웹캠    51장
    +
CelebA Live      200장 (샘플링, 1:1 균형)
    +
CelebA 공격류    각 300장 유지
```